# 일단 ai 헬멧 감지 

In [1]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 주피터 노트북용 비동기 패치 (에러 방지)
nest_asyncio.apply()

# 오라클 클라이언트 초기화
try:
    oracledb.init_oracle_client()
except Exception as e:
    pass

# FastAPI 앱 생성
app = FastAPI()

# YOLO 모델 로딩
model_path = "data/best (sDUDU).pt"

model = None
try:
    if os.path.exists(model_path):
        model = YOLO(model_path)
        print(f"ai 모델 로딩 성공({model+path})")
        print(f"감지 가능목록(names): {model.names}")
    else:
        print(f"파일이 없음:{os.path.abspath(model_path)}")
except Exception as e:
    print(f"모델 로딩 중 에러: {e}")

# db 연결 함수
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"db접속 에러 : {e}")
        return None

# 메인 기능 : 자바가 요청 -> 카메라 켜고 감지 -> db 저장
@app.get("/helmet-check")
def helmet_check(kickboard_id: str):
    print(f"요청도착 킥보드id : {kickboard_id}")

    # ai 감지 로직
    helmet_status = "미착용"
    is_helmet_detected = False
    detected_objects = []

    if model:
        # 동영상 파일 경로
        video_path = "data/야간1(헬멧o).mp4"
        print(f"동영상({video_path})을 분석합니다.")
        
        if os.path.exists(video_path):
            # stream=True : 한번에 처리 x, 프레임별로 처리
            # max_det=1 : 한 프레임당 1개만 감지
            # vid_stride=30 : 모든 프레임x 30프레임마다 1번씩 검사
            results = model.predict(source=video_path, save=True, conf=0.5, vid_stride=30)

            # 결과 분석
            for result in results:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    class_name = model.names[cls_id]

                    # 중복 제거해 리스트에 담기
                    if class_name not in detected_objects:
                        detected_objects.append(class_name)
                        print(f"ai가본것:{class_name}")

                    # 헬멧 감지 여부 체크
                    if 'helmet' in class_name.lower():
                        is_helmet_detected = True
                        helmet_status = "착용"
                        # 헬멧을 찾으면 더 검사 X 
                        break
                if is_helmet_detected: break
        else:
            print(f"영상 파일이 없음:{video_path}")
    else:
        print(f"모델이 없음")

모델 로딩 중 에러: name 'path' is not defined


In [2]:
import os  # ★ 이 친구가 없으면 path 관련 에러가 납니다!
from ultralytics import YOLO

# 경로 설정 (아까 만든 data 폴더)
model_path = "data/best.pt"

# 디버깅용: 현재 폴더 위치와 파일이 진짜 있는지 확인
print(f"📂 현재 작업 위치: {os.getcwd()}")

if os.path.exists(model_path):  # ★ 여기에 os.path 라고 정확히 써야 합니다!
    try:
        model = YOLO(model_path)
        print(f"✅ 모델 로딩 성공! ({model_path})")
        print(f"📋 감지 목록: {model.names}")
    except Exception as e:
        print(f"💥 모델 파일은 있는데 로딩 실패: {e}")
else:
    # 파일이 없을 때 절대경로를 보여줌 (찾기 쉽게)
    print(f"💥 파일을 못 찾겠어요! 여기 있는지 확인해보세요: {os.path.abspath(model_path)}")
    model = None

📂 현재 작업 위치: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python
💥 파일을 못 찾겠어요! 여기 있는지 확인해보세요: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\data\best.pt


In [3]:
import os

print("📂 'data' 폴더 안에 있는 실제 파일 목록:")
print("-" * 30)

try:
    files = os.listdir("data") # data 폴더 안을 들여다봅니다.
    for f in files:
        print(f"👉 발견된 파일: {f}")
        
    if not files:
        print("텅 비어있는데요? 😅")
        
except Exception as e:
    print(f"💥 에러! 'data'라는 폴더 자체가 없는 것 같아요. (현재 위치에 있는 폴더들: {os.listdir('.')})")

📂 'data' 폴더 안에 있는 실제 파일 목록:
------------------------------
👉 발견된 파일: 299823_tiny.mp4
👉 발견된 파일: 39183-421020269_tiny.mp4
👉 발견된 파일: best (sDUDU).pt
👉 발견된 파일: minha.mp4
👉 발견된 파일: minha2.mp4
👉 발견된 파일: results.csv
👉 발견된 파일: results.png
👉 발견된 파일: test1.mp4
👉 발견된 파일: testvideo.mp4
👉 발견된 파일: testvideo1.mp4
👉 발견된 파일: testvideo_demo.mp4
👉 발견된 파일: 야간1(노헬멧).mp4
👉 발견된 파일: 야간1(헬멧o).mp4
👉 발견된 파일: 야간2(헬멧o).mp4


In [4]:
import os  
from ultralytics import YOLO

# 모델 경로(위치)
model_path = "data/best (sDUDU).pt"

print(f"📂 현재 위치: {os.getcwd()}")

# [수정] 그냥 path.exists가 아니라 'os.path.exists'가 맞습니다!
if os.path.exists(model_path):
    try:
        model = YOLO(model_path)
        print(f"✅ 모델 로딩 성공! ({model_path})")
        print(f"📋 클래스 목록(names): {model.names}") 
    except Exception as e:
        print(f"💥 파일은 있는데 로딩 실패: {e}")
else:
    # [수정] 여기도 path.abspath가 아니라 'os.path.abspath'
    print(f"💥 파일을 못 찾겠어요: {os.path.abspath(model_path)}")
    model = None

📂 현재 위치: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python
✅ 모델 로딩 성공! (data/best (sDUDU).pt)
📋 클래스 목록(names): {0: 'no_helmet', 1: 'helmet'}


In [7]:
# ==========================================
# [1] 필요한 도구(라이브러리)들을 가져오는 구역
# ==========================================
import uvicorn              # 서버를 실행시켜주는 도구 (웹 서버)
import nest_asyncio         # 주피터 노트북에서 서버가 에러 없이 돌게 해주는 패치
from fastapi import FastAPI # 웹 페이지 요청을 받아주는 핵심 도구
import oracledb             # 오라클 DB와 대화하기 위한 전화기
import datetime             # 날짜와 시간을 다루는 도구
import os                   # 파일 경로(폴더 위치)를 확인하는 도구
from ultralytics import YOLO # AI(YOLO) 모델을 사용하는 도구

# 주피터 노트북은 원래 비동기 작업(서버 돌리기 등)을 막아놓는데, 
# 이걸 풀어주는 코드입니다. (이거 없으면 에러남!)
nest_asyncio.apply()

# ==========================================
# [2] 기본 설정 (오라클 & 웹 서버)
# ==========================================

# 오라클 클라이언트(접속 프로그램) 초기화
# 가끔 설치 안 된 PC에서 에러가 나서 try-except로 감싸둠
try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

# FastAPI 앱(서버 본체) 생성
app = FastAPI()

# ==========================================
# [3] AI 모델 로딩 (가장 중요한 뇌 부분)
# ==========================================

# 우리가 사용할 학습된 모델 파일 경로
# 주의: 파일명 띄어쓰기, 괄호까지 정확해야 함!
model_path = "data/best (sDUDU).pt" 
model = None # 일단 빈 변수로 시작

# 파일이 진짜 있는지 확인하고 로딩
if os.path.exists(model_path):
    # 모델 파일을 읽어와서 메모리에 올림
    model = YOLO(model_path)
    print(f"✅ AI 모델 로딩 성공! ({model_path})")
    print(f"📋 이 모델이 아는 것들: {model.names}") 
    # {0: 'no_helmet', 1: 'helmet'} 확인됨
else:
    print(f"💥 모델 파일이 없어요! 경로 확인필요: {model_path}")

# ==========================================
# [4] DB 연결 함수 (전화기 들기)
# ==========================================
def get_db_connection():
    try:
        # DB 접속 정보 입력 (주소, 포트, 서비스이름)
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        
        # 실제 연결 시도 (아이디, 비밀번호)
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn # 연결 성공하면 전화기(conn)를 반환
    except Exception as e:
        print(f"❌ DB 접속 실패: {e}")
        return None

# ==========================================
# [5] 메인 기능: 헬멧 감지 요청 처리
# ==========================================
# 자바가 "http://IP:8001/helmet-check?kickboard_id=..." 로 접속하면 이 함수가 실행됨
@app.get("/helmet-check")
def helmet_check(kickboard_id: str):
    
    # 1. 요청이 왔음을 알림
    print(f"📡 [요청 도착] 킥보드 ID: {kickboard_id}")
    
    # --- [A] AI 감지 로직 (영상 분석) ---
    helmet_status = "미착용"     # 일단 '미착용'이라고 가정 (안전빵)
    is_helmet_detected = False   # 헬멧 쓴 사람 발견했니? (아직 못 봄)
    detected_list = []           # 로그 확인용 리스트

    if model: # 모델이 정상적으로 로딩되었을 때만 실행
        
        # 분석할 테스트 영상 파일 경로
        video_path = "data/야간1(헬멧o).mp4" 
        print(f"🎬 영상 분석 시작: {video_path}")
        
        if os.path.exists(video_path):
            # predict: 예측해라! 
            # source: 영상파일, save: 결과사진 저장, conf: 50% 이상 확실할 때만
            # vid_stride=30: 속도를 위해 30프레임마다 1번씩만 검사 (매우 중요!)
            results = model.predict(source=video_path, save=True, conf=0.5, vid_stride=30, verbose=False)
            
            # 분석 결과(results)를 하나씩 까봅니다
            for result in results:
                for box in result.boxes:
                    # box.cls[0]: 감지된 물체의 번호 (0 또는 1)
                    cls_id = int(box.cls[0])           
                    # model.names[cls_id]: 번호를 이름으로 바꿈 ('helmet' 등)
                    class_name = model.names[cls_id]   
                    
                    # 로그 찍기 (중복 방지)
                    if class_name not in detected_list:
                        detected_list.append(class_name)
                        print(f"🧐 AI 발견: {class_name}")

                    # ★ 핵심 규칙 ★
                    # 모델이 알려준 번호가 1번(helmet)이면 '착용'으로 인정!
                    if cls_id == 1: 
                        is_helmet_detected = True
                        helmet_status = "착용"
                        
            print(f"📊 최종 판단 결과: {helmet_status}")
            
        else:
            print(f"💥 영상 파일이 없어서 분석을 못해요: {video_path}")
    
    # --- [B] DB 저장 로직 ---
    result_msg = "실패"
    conn = get_db_connection() # DB 접속
    
    if conn:
        try:
            cursor = conn.cursor() # 쿼리 날릴 준비
            
            # 현재 시간 구하기
            now = datetime.datetime.now()
            # RIDE_ID 만들기 (예: RIDE_20250115143000)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            # 점수 계산: 헬멧 썼으면 10점, 안 썼으면 -5점 벌점
            score_cg = 10 if is_helmet_detected else -5
            # 미착용 횟수: 썼으면 0, 안 썼으면 1
            no_helmet_cnt = 0 if is_helmet_detected else 1
            
            # DB에 넣을 데이터 포장하기 (순서 중요!)
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,   
                "USER010",      # 회원 ID (실제 DB에 있는 사람)
                kickboard_id,  # 자바에서 받아온 킥보드 ID (DD010)
                now, now,      # 시작시간, 종료시간
                12.5,          # 주행시간 (임시값)
                no_helmet_cnt, # 미착용 횟수
                score_cg,      # 점수
                "P"            # 운행상태 (Parking 등 규칙에 맞는 값)
            )
            
            # 쿼리 실행 및 저장(commit)
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 점수: {score_cg})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러 발생: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close() # 전화 끊기 (필수)
            
    # 자바(안드로이드)에게 결과를 JSON 형태로 돌려줌
    return {
        "result": result_msg,
        "kickboard_id": kickboard_id,
        "helmet_check": helmet_status,
        "detected": detected_list
    }

# ==========================================
# [6] 서버 실행 구역
# ==========================================
if __name__ == "__main__":
    # 포트 8001번으로 서버를 엽니다.
    # host="0.0.0.0"은 외부(팀원 컴퓨터) 접속을 허용한다는 뜻
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 로딩 성공! (data/best (sDUDU).pt)
📋 이 모델이 아는 것들: {0: 'no_helmet', 1: 'helmet'}


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드 ID: DD010
🎬 영상 분석 시작: data/야간1(헬멧o).mp4
WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict
🧐 AI 발견: helmet
🧐 AI 발견: no_helmet
📊 최종 판단 결과: 착용
💾 DB 저장 완료! (ID: RIDE_20260115160113, 점수: 10)
INFO:     192.168.219.166:58505 - "GET /helmet-check?kickboard_id=DD010 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [13]:
import os

print("📂 'data' 폴더 파일 목록:")
print("-" * 30)
# data 폴더 안의 내용을 보여줘!
files = os.listdir("data") 
for f in files:
    print(f)

📂 'data' 폴더 파일 목록:
------------------------------
299823_tiny.mp4
39183-421020269_tiny.mp4
best (sDUDU).pt
minha.mp4
minha2.mp4
results.csv
results.png
test1.mp4
testvideo.mp4
testvideo1.mp4
testvideo_demo.mp4
야간1(노헬멧)편집.mp4
야간1(헬멧o)편집.mp4
야간2(헬멧o)편집.mp4
주간1(노헬멧).mp4
주간1(헬멧o).mp4


# 점수 계산식 적용버전(비율 90%)

In [14]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/주간1(헬멧o).mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 30:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 30) // 30
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/주간1(헬멧o).mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict5
헬멧 착용 O (+0.02점
💾 DB 저장 완료! (ID: RIDE_20260115172143, 유저: USER001, 점수: 0.02)
INFO:     192.168.219.166:63475 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [17]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/주간1(노헬멧).mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 5:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 5) // 5
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/주간1(노헬멧).mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict7
미착용 확정!(-3점
💾 DB 저장 완료! (ID: RIDE_20260115172700, 유저: USER001, 점수: -3.00)
INFO:     192.168.219.166:55410 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [18]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/야간1(헬멧o)편집.mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 5:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 5) // 5
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/야간1(헬멧o)편집.mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict8
헬멧 착용 O (+0.02점
💾 DB 저장 완료! (ID: RIDE_20260115172836, 유저: USER001, 점수: 0.02)
INFO:     192.168.219.166:55415 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [19]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 주피터(파이썬)에서 막은 비동기방식 해제하는 코드
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/야간1(노헬멧)편집.mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 3:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 3) // 3
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/야간1(노헬멧)편집.mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict9
미착용 확정!(-5점
💾 DB 저장 완료! (ID: RIDE_20260115172948, 유저: USER001, 점수: -5.00)
INFO:     192.168.219.166:55417 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [3]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 주피터(파이썬)에서 막은 비동기방식 해제하는 코드
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/야간2(헬멧o)편집.mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 3:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 3) // 3
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [12504]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/야간2(헬멧o)편집.mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict10
헬멧 착용 O (+0.02점
💾 DB 저장 완료! (ID: RIDE_20260116110249, 유저: USER001, 점수: 0.02)
INFO:     192.168.219.235:52299 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [12504]


## 데이터베이스 조회

In [ ]:
!pip install

In [2]:
# db 조회용
import oracledb
import pandas as pd

try:
    oracledb.init_oracle_client()
except:
    pass

dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)

# 최근 데이터만 조회하기
sql = """
    SELECT RIDE_ID, USER_ID, KICKBOARD_ID, 
           HELMET_TM as "주행시간",
           NOHELMET_CNT as "미착용",
           SCORE_CG as "점수",
           RIDE_ST as "상태"
    FROM TB_RIDE
    ORDER BY START_DT DESC
"""

try:
    df = pd.read_sql(sql, conn)
    display(df.head(10))
except Exception as e:
    print(f"에러{e}")
finally:
    conn.close()

C:\Users\smhrd\AppData\Local\Temp\ipykernel_12504\3779785910.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,RIDE_ID,USER_ID,KICKBOARD_ID,주행시간,미착용,점수,상태
0,RIDE_20260115172948,USER001,DD010,12.00,12,-5,P
1,RIDE_20260115172836,USER001,DD010,12.00,0,0,P
2,RIDE_20260115172700,USER001,DD010,11.00,11,-3,P
3,RIDE_20260115172413,USER001,DD010,11.00,11,0,P
4,RIDE_20260115172143,USER001,DD010,11.00,0,0,P
5,RIDE_20260115171240,USER001,DD010,158.00,57,0,P
6,RIDE_20260115164600,USER001,DD010,142.00,48,0,P
7,RIDE_20260115160113,USER010,DD010,12.50,0,10,P
8,JUPITER_RIDE000,USER010,DD010,12.50,3,10,E
9,RIDE026,USER006,DD006,11.57,0,28,C


# 더미데이터 넣기

## 📚 핵심 함수 & 문법 사전
> **더미 데이터 생성 코드**에서 사용된 주요 파이썬 기능 정리입니다.  
> 면접 질문 대비나 코드 리뷰 시 참고하세요!

---

### 1. `f"{변수:03d}"` (f-string 포맷팅)
문자열 앞에 `f`를 붙이면, 중괄호 `{}` 안에 변수를 바로 넣을 수 있는 파이썬의 강력한 기능입니다.
* **`:03d`의 의미:** 숫자를 출력할 때 **"3자리를 확보하고, 빈 곳은 0으로 채워라"**
* **예시:**
    * `1` → `"001"`
    * `20` → `"020"`

### 2. `random.choice(리스트)`
주어진 리스트 안에 있는 데이터 중 **아무거나 하나를 무작위로 쏙 뽑아주는** 함수입니다.
* **비유:** 제비뽑기 상자에서 종이 하나 꺼내기

### 3. `random.random() < 0.7`
`random.random()`은 `0.000...` 부터 `0.999...` 사이의 실수를 랜덤으로 생성합니다.
* **확률 구현의 핵심:**
    * 나온 숫자가 0.7보다 작을 확률은 수학적으로 **70%**입니다.
    * 따라서 **"70% 확률로 실행해라(True)"** 라는 로직을 만들 때 가장 많이 쓰이는 공식입니다.

### 4. `datetime.timedelta(days=...)`
`time` + `delta`(차이/간격)의 합성어입니다. 날짜나 시간의 **"간격"**을 의미합니다.
* **활용:** 날짜 계산(더하기/빼기)에 사용됩니다.
* **예시:** `현재시간 - timedelta(days=3)`  
  👉 "현재로부터 3일 전(과거) 날짜"가 계산됨

### 5. `round(숫자, 2)`
소수점이 길게 나올 때 예쁘게 다듬어주는 함수입니다.
* **의미:** 소수점 **둘째 자리까지** 남기고 반올림해라.
* **예시:** `round(3.141592, 2)` → `3.14`

In [5]:
import oracledb
import random
import datetime

# db 연결
try:
    # 오라클 클라이언트 초기화 시키는 함수
    oracledb.init_oracle_client()
except:
    pass # -> 이미 초기화 했거나 필요없으면 무시하고 넘어감
dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
# cursor : DB에 SQL 명령을 배달하고 결과 가져오는 객체
cursor = conn.cursor()
print("더미 데이터 넣기 시작")

# 더미 리스트 생성 -> 리스트 컴프리헨션 (List Comprehension)
# 반복문을 한 줄로 줄여서 리스트를 만드는 기능.
# f"USER{i:03d}" : f-string 포맷팅
#     - {i} : 숫자 변수 i를 넣음
#     - :03d : 숫자를 3자리로 맞추고, 빈 자리는 0으로 채워라 (1->001)
user_list = [f"USER{i:03d}" for i in range(1, 21)]
kickboard_list = [f"DD{i:03d}" for i in range(1, 21)]
print(f"준비한 유저 : {len(user_list)}명 / 킥보드 : {len(kickboard_list)}대")

# 데이터 100개 생성하는 루프
for i in range(100):
    # 랜덤 뽑기 -> choice : 리스트 안에서 아무거나 하나를 무작위로 뽑음.
    u_id = random.choice(user_list)
    k_id = random.choice(kickboard_list)

    # 날짜 랜덤(최근 30일) // randint(a, b) : a와 b 사이의 정수를 랜덤으로 뽑음.
    days_ago = random.randint(0, 30)
    # datetime.now() : 현재 컴퓨터의 날짜와 시간을 가져옴.
    # datetime.timedelta() : 시간의 차이(간격)을 나타냄. (덧셈/뺄셈 가능) => 현재 시간 - (몇일+몇시간+몇분) = 과거의 랜덤한 시간
    start_time = datetime.datetime.now() - datetime.timedelta(
        days=days_ago,
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

    # 주행 시간 : 3분(180초) ~ 30분(1800초) 사이 랜덤
    duration_sec = random.randint(180, 1800)

    # 종료 시간 = 시작 시간 + 주행 시간(초)
    end_time = start_time + datetime.timedelta(seconds=duration_sec)

    # 더미 데이터 시나리오 
    # 모범 사례 (70%) vs 위반 사례 (30%)
    # random.random() : 0.0 ~ 1.0 사이의 실수(소수)를 랜덤으로 뽑음
    is_good_driver = random.random() < 0.7

    if is_good_driver:
        # 헬멧 착용률 90~100%
        # random.uniform(a, b) : a와 b 사이의 실수(float)를 랜덤으로 뽑음
        helmet_tm = int(duration_sec * random.uniform(0.9, 1.0))
        nohelmet_cnt = 0

        # 점수 : 분당 0.1점 가산
        score = (duration_sec / 60) * 0.1
    else:
        # 위반 -> 헬멧 착용률 0~40%
        helmet_tm = int(duration_sec * random.uniform(0.0, 0.4))
        nohelmet_cnt = duration_sec - helmet_tm
        # 점수 : -2점 ~ -10점 랜덤으로 감점
        score = random.randint(-10, -2)

    # RIDE_ID 생성
    # strftime("%Y%m%d") : 날짜 데이터를 "20250116" 같은 문자열로 변환 (String Format Time)
    # f"_{i:03d}" : 반복 횟수(i)를 붙여 겹치지 않게 함.
    ride_id = "DUMMY_" + start_time.strftime("%Y%m%d") + f"_{i:03d}"

    # DB 저장
    # :1, :2 등은 파이썬 변수를 넣을 구멍(Placeholder)임
    # 보안상 안전하고 속도가 빠름
    sql = """
        INSERT INTO TB_RIDE (
            RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT,
            HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
        )
        VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
    """

    # 튜플()로 묶어서 순서대로 데이터를 준비
    data = (
        ride_id, u_id, k_id, start_time, end_time, helmet_tm, nohelmet_cnt, round(score, 2), "P"        
    )

    try:
        # 쿼리 실행, 커서에게 
        cursor.execute(sql, data)
    except Exception as e:
        # 에러가 나면 이유만 출력하고 진행
        print(f"에러({ride_id}) : {e}")

# w저장
conn.commit()
conn.close()
print("데이터 저장 성공")

더미 데이터 넣기 시작
준비한 유저 : 20명 / 킥보드 : 20대
에러(DUMMY_20260106_001) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20251220_004) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20251229_007) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20251228_015) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20251223_017) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20251218_022) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20260109_023) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20260116_026) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20260105_027) : ORA-01438: value larger than specified precision allowed for this column
에러(DUMMY_20251222_032) : ORA-01438: value larger than specified precision allowe

### 위쪽 오류는 데이터베이스의 시간초의 범위가 코드로 설정한 범위보다 작아서 나는 오류 -> 설정 다시

In [6]:
import oracledb
import random
import datetime

# 1. DB 연결
try:
    oracledb.init_oracle_client()
except:
    pass

dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
cursor = conn.cursor()

print("🚚 안전한 범위의 데이터로 재생성 시작...")

# 데이터 준비
user_list = [f"USER{i:03d}" for i in range(1, 21)] 
kickboard_list = [f"DD{i:03d}" for i in range(1, 21)]
data_buffer = [] 

for i in range(100):
    u_id = random.choice(user_list)
    k_id = random.choice(kickboard_list)
    
    days_ago = random.randint(0, 30)
    start_time = datetime.datetime.now() - datetime.timedelta(
        days=days_ago, hours=random.randint(0, 23), minutes=random.randint(0, 59)
    )
    
    # ★ [수정] 1800초 -> 900초 (15분)으로 줄임!
    # DB 컬럼이 3자리(999)까지만 받을 수 있어서 1000을 넘기면 안 됨
    duration_sec = random.randint(180, 900) 
    
    end_time = start_time + datetime.timedelta(seconds=duration_sec)
    
    is_good_driver = random.random() < 0.7 
    
    if is_good_driver:
        helmet_tm = int(duration_sec * random.uniform(0.9, 1.0))
        nohelmet_cnt = 0
        score = (duration_sec / 60) * 0.1
    else:
        helmet_tm = int(duration_sec * random.uniform(0.0, 0.4))
        nohelmet_cnt = duration_sec - helmet_tm
        score = random.randint(-9, -2) # 혹시 몰라 -10(두자리) 대신 -9(한자리)로 안전하게

    ride_id = "BATCH_" + start_time.strftime("%Y%m%d") + f"_{i:03d}"
    
    row = (
        ride_id, u_id, k_id, start_time, end_time,
        helmet_tm, nohelmet_cnt, round(score, 2), "P"
    )
    data_buffer.append(row)

# 한 방에 전송
sql = """
    INSERT INTO TB_RIDE (
        RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
        HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
    ) 
    VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
"""

try:
    cursor.executemany(sql, data_buffer)
    conn.commit() 
    print(f"✅ 성공! {len(data_buffer)}개 데이터가 에러 없이 저장되었습니다.")
except Exception as e:
    print(f"💥 여전히 에러 발생: {e}")
finally:
    conn.close()

🚚 안전한 범위의 데이터로 재생성 시작...
✅ 성공! 100개 데이터가 에러 없이 저장되었습니다.


# 경고음 + 안전점수 계산 로직

In [2]:
!pip install ultralytics

In [3]:
!pip install pygame

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ----------------------- ---------------- 6.3/10.6 MB 67.4 MB/s eta 0:00:01
   ---------------------------------- ----- 9.2/10.6 MB 22.6 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 22.1 MB/s eta 0:00:00


In [5]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO
import pygame # [추가] 소리 재생용
import time   # [추가] 쿨타임 계산용

# 주피터(파이썬)에서 막은 비동기방식 해제하는 코드
nest_asyncio.apply()

# 1. 오디오 초기화 [추가됨]
try:
    pygame.mixer.init()
    # 경고음 파일 경로 확인! (없으면 에러 로그만 출력하고 멈추지 않음)
    SOUND_PATH = "data/MP_Woop Woop.mp3" 
    if os.path.exists(SOUND_PATH):
        alert_sound = pygame.mixer.Sound(SOUND_PATH)
        print("🔊 경고음 시스템 준비 완료")
    else:
        alert_sound = None
        print("🔇 경고음 파일 없음 (무음 모드)")
except Exception as e:
    alert_sound = None
    print(f"🔇 오디오 장치 에러: {e}")

# Oracle 클라이언트 초기화
try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결 함수
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# API 엔드포인트
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] 변수 초기화
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # 경고음 쿨타임 변수 [추가됨]
    last_alert_time = 0 
    ALERT_INTERVAL = 3.0 # 3초에 한 번만 울림

    # ★ 영상 경로 (테스트용)
    video_path = "data/야간노헬멧.mp4"

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        
        # YOLO 예측 시작
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            current_frame_status = "none" # 현재 프레임 상태 (helmet / no_helmet)

            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet (모델에 따라 다를 수 있음. 확인필요!)
                
                # [주의] 본인 모델의 클래스 번호 확인 필요 (0번이 no-helmet인지, 1번인지)
                # 여기서는 기존 코드대로 0: no-helmet, 1: helmet 이라고 가정함
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    current_frame_status = "helmet"
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    current_frame_status = "no_helmet"
                    break
            
            if found_something:
                total_seconds += 1
                
                # -----------------------------------------------------
                # [★ 핵심 추가] 실시간 경고음 로직
                # -----------------------------------------------------
                if current_frame_status == "no_helmet":
                    current_time = time.time()
                    # 소리 파일이 있고 && 쿨타임(3초)이 지났으면 재생
                    if alert_sound and (current_time - last_alert_time > ALERT_INTERVAL):
                        alert_sound.play()
                        last_alert_time = current_time
                        print(f"   ㄴ 🔊 경고음 발사! (현재 {total_seconds}초 경과)")
                # -----------------------------------------------------

        # [B] 점수 계산 로직 (기존 유지)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (60% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"결과: 헬멧 착용 O (+{plus_score:.2f}점)")

            # (2) 미착용 모드 (60% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 3:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (3초마다? 기존 로직은 3개 프레임마다로 보임)
                    extra = (no_helmet_count - 3) // 3
                    if extra > 0:
                        minus_score += (extra * 1)
                
                final_score -= minus_score
                print(f"결과: 미착용 확정! (-{minus_score}점)")
            else:
                helmet_status = "판독애매"
                print("결과: 비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 모델이 없거나 영상 경로가 잘못됨")

    # [C] DB 저장 로직 (기존 유지)
    result_msg = "실패"
    conn = get_db_connection()
    new_ride_id = "ERROR"

    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    
                user_id,        # 파라미터로 받은 값
                kickboard_id,   # 파라미터로 받은 값
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id,
        "score": round(final_score, 2),
        "status": helmet_status,
        "alert_count": no_helmet_count # 참고용으로 추가
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

🔊 경고음 시스템 준비 완료
✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [6168]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD001 / 사용자: USER001
🎬 분석 시작: data/야간노헬멧.mp4
   ㄴ 🔊 경고음 발사! (현재 1초 경과)
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict2
결과: 미착용 확정! (-5점)
💾 DB 저장 완료! (ID: RIDE_20260119122150)
INFO:     127.0.0.1:11773 - "GET /helmet-check?kickboard_id=DD001&user_id=USER001 HTTP/1.1" 200 OK
INFO:     127.0.0.1:11773 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [6168]


# 더미데이터 더 채워넣기 (60일치)

In [8]:
import oracledb
import random
import datetime

try:
    oracledb.init_oracle_client()
except:
    pass

dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
cursor = conn.cursor()

print("기존 데이터 삭제중")
cursor.execute("DELETE FROM TB_RIDE")
conn.commit()
print("60일치 데이터 생성 시작")

# 더미 데이터 생성
user_list = [f"USER{i:03d}" for i in range(1, 26)]
kickboard_list = [f"DD{i:03d}" for i in range(1, 21)]
# DB에 한번에 넣기 위해 데이터를 모을 리스트
data_buffer = []

# 60일 전부터 오늘까지 날짜를 반복
# range(60, -1, -1) -> 60, 59, 58, ... , 1, 0 순서로 줄어든다.
for days_ago in range(60, -1, -1):
    # 현재 계산 중인 날짜 (오늘 - days_ago)
    current_date = datetime.datetime.now() - datetime.timedelta(days=days_ago)

    # 디테일로 금 토 일은 더 많이 탄다고 가정
    if current_date.weekday() >= 4:
        daily_count = random.randint(15, 30)
    else:
        daily_count = random.randint(8, 15)

    # ★★★ [성장 그래프의 비밀] ★★★
    # progress는 시간이 지날수록 0.0에서 1.0으로 변합니다.
    # 60일 전(과거) -> (60-60)/60 = 0.0 (0%)
    # 30일 전(중간) -> (60-30)/60 = 0.5 (50%)
    # 0일 전(오늘) -> (60-0)/60 = 1.0 (100%)
    progress = (60 - days_ago) / 60

    # 착한 운전자가 될 확률 계산
    good_driver_prob = 0.1 + (0.85 * progress)

    # 하루치 데이터
    for _ in range(daily_count):
        u_id = random.choice(user_list)
        k_id = random.choice(kickboard_list)

        # 주행시간 랜덤
        ride_time = current_date.replace(
            hour = random.randint(7, 23),
            minute = random.randint(0, 59),
            second = random.randint(0, 59)
        )

        # 주행시간 (5 ~ 30분 사이 랜덤)
        duration_sec = random.randint(300, 1800)
        # 주행 종료시간
        end_time = ride_time + datetime.timedelta(seconds=duration_sec)

        # 위에서 계산한 확률로 좋은/나쁜 운전자 결정
        is_good = random.random() < good_driver_prob

        if is_good:
            # 좋은 운전자 : 전체 주행 시간의 95~100% 헬멧 착용
            helmet_tm = int(duration_sec * random.uniform(0.95, 1.0))
            nohelmet_cnt = 0
            score = random.randint(10, 20)
        else:
            # 나쁜 운전자 : 헬멧 거의 안씀. 점수 깎임
            helmet_tm = int(duration_sec * random.uniform(0.0, 0.2))
            nohelmet_cnt = duration_sec - helmet_tm
            score = random.randint(-10, -5)

        # 주행 id 생성
        ride_id = "BATCH_" + ride_time.strftime("%Y%m%d%H%M%S") + f"_{random.randint(100,999)}"
        # DB에 넣을 한 줄 완성
        row = (ride_id, u_id, k_id, ride_time, end_time, helmet_tm, nohelmet_cnt, score, "P")
        data_buffer.append(row)

# db에 데이터 전송 (insert)
sql = """
    INSERT INTO TB_RIDE (
        RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
        HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
    ) 
    VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
"""

try:
    # 만든 데이터 덩어리를 한번에 전송
    cursor.executemany(sql, data_buffer)
    conn.commit()
    print(f"성공 60일치({len(data_buffer)}건) 데이터 저장 완료")
except Exception as e:
    print(f"에러 : {e}")
finally:
    conn.close()

기존 데이터 삭제중
60일치 데이터 생성 시작
에러 : ORA-02291: integrity constraint (CAMPUS_25IS_GA2_P2_4.FK_TB_RIDE_USER) violated - parent key not found


In [10]:
import oracledb      
import random        
import datetime      

# 1. DB 연결
try:
    oracledb.init_oracle_client()
except:
    pass

dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
cursor = conn.cursor()

# 2. 실제 회원/킥보드 ID 조회 (FK 오류 방지)
print("🔍 실제 데이터 조회 중...")
try:
    cursor.execute("SELECT USER_ID FROM TB_USER")
    user_list = [row[0] for row in cursor.fetchall()]
    
    cursor.execute("SELECT KICKBOARD_ID FROM TB_KICKBOARD")
    kickboard_list = [row[0] for row in cursor.fetchall()]
    
    print(f"   ㄴ 회원 {len(user_list)}명 / 킥보드 {len(kickboard_list)}대 확인 완료")
except Exception as e:
    print(f"⚠️ 데이터 조회 실패 (임시 ID 사용): {e}")
    user_list = [f"USER{i:03d}" for i in range(1, 26)]
    kickboard_list = [f"DD{i:03d}" for i in range(1, 21)]

if len(user_list) == 0:
    print("❌ 에러: 회원이 한 명도 없습니다. 회원가입을 먼저 해주세요.")
    conn.close()
    exit()

# 3. 데이터 초기화
print("🧹 기존 데이터 삭제 중...")
cursor.execute("DELETE FROM TB_RIDE") 
conn.commit()

print("✨ 60일치 데이터 생성 시작 (숫자 크기 조정 완료)... 🚀")
data_buffer = []

for days_ago in range(60, -1, -1): 
    
    current_date = datetime.datetime.now() - datetime.timedelta(days=days_ago)
    
    if current_date.weekday() >= 4: 
        daily_count = random.randint(15, 30)
    else:
        daily_count = random.randint(8, 15)
    
    progress = (60 - days_ago) / 60
    good_driver_prob = 0.1 + (0.85 * progress)
    
    for _ in range(daily_count):
        u_id = random.choice(user_list)
        k_id = random.choice(kickboard_list)
        
        ride_time = current_date.replace(
            hour=random.randint(7, 23), 
            minute=random.randint(0, 59),
            second=random.randint(0, 59)
        )
        
        # ★★★ [수정된 부분] ★★★
        # 기존: 300 ~ 1800 (1800은 NUMBER(3)에 안 들어감)
        # 수정: 180 ~ 900 (최대 900초 = 15분, 3자리를 넘지 않음!)
        duration_sec = random.randint(180, 900) 
        
        end_time = ride_time + datetime.timedelta(seconds=duration_sec)
        
        is_good = random.random() < good_driver_prob
        
        if is_good:
            helmet_tm = int(duration_sec * random.uniform(0.95, 1.0))
            nohelmet_cnt = 0
            score = random.randint(10, 20) 
        else:
            helmet_tm = int(duration_sec * random.uniform(0.0, 0.2))
            nohelmet_cnt = duration_sec - helmet_tm
            score = random.randint(-10, -5)

        ride_id = "BATCH_" + ride_time.strftime("%Y%m%d%H%M%S") + f"_{random.randint(100,999)}"
        
        row = (ride_id, u_id, k_id, ride_time, end_time, helmet_tm, nohelmet_cnt, score, "P")
        data_buffer.append(row)

sql = """
    INSERT INTO TB_RIDE (
        RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
        HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
    ) 
    VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
"""

try:
    cursor.executemany(sql, data_buffer)
    conn.commit() 
    print(f"✅ 성공! 오류 없이 {len(data_buffer)}건이 저장되었습니다.")
except Exception as e:
    print(f"💥 에러 발생: {e}")
finally:
    conn.close()

🔍 실제 데이터 조회 중...
   ㄴ 회원 25명 / 킥보드 56대 확인 완료
🧹 기존 데이터 삭제 중...
✨ 60일치 데이터 생성 시작 (숫자 크기 조정 완료)... 🚀
✅ 성공! 오류 없이 1004건이 저장되었습니다.


In [11]:
import oracledb
import random
import datetime

# DB 연결
try:
    oracledb.init_oracle_client()
except:
    pass
dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
cursor = conn.cursor()

# -----------------------------------------------------------
# [1] 실제 회원/킥보드 ID 조회
# -----------------------------------------------------------
print("🔍 데이터 준비 중...")
try:
    cursor.execute("SELECT USER_ID FROM TB_USER")
    user_list = [row[0] for row in cursor.fetchall()]
    cursor.execute("SELECT KICKBOARD_ID FROM TB_KICKBOARD")
    kickboard_list = [row[0] for row in cursor.fetchall()]
except:
    user_list = [f"USER{i:03d}" for i in range(1, 26)]
    kickboard_list = [f"DD{i:03d}" for i in range(1, 21)]

if len(user_list) == 0:
    print("❌ 에러: 회원이 없습니다.")
    conn.close()
    exit()

# -----------------------------------------------------------
# [2] 초기화
# -----------------------------------------------------------
print("🧹 기존 데이터 삭제...")
cursor.execute("DELETE FROM TB_RIDE")
conn.commit()

print("✨ '발표용 최적화' 데이터 생성 시작 (단위 수정됨)... 🚀")
data_buffer = []

# -----------------------------------------------------------
# [3] 60일치 생성 (단위: 횟수, 시작점: 40%)
# -----------------------------------------------------------
for days_ago in range(60, -1, -1):
    
    current_date = datetime.datetime.now() - datetime.timedelta(days=days_ago)
    
    # 주말 가중치
    if current_date.weekday() >= 4:
        daily_count = random.randint(15, 25)
    else:
        daily_count = random.randint(8, 12)
    
    # ★ 성장 로직 수정 ★
    progress = (60 - days_ago) / 60
    # 40%에서 시작해서 95%까지 성장 (너무 바닥을 기지 않게!)
    good_driver_prob = 0.4 + (0.55 * progress)
    
    for _ in range(daily_count):
        u_id = random.choice(user_list)
        k_id = random.choice(kickboard_list)
        
        # 시간 랜덤
        ride_time = current_date.replace(hour=random.randint(7, 23), minute=random.randint(0, 59))
        duration_sec = random.randint(300, 900) # 5분~15분
        end_time = ride_time + datetime.timedelta(seconds=duration_sec)
        
        # 운전 성향
        is_good = random.random() < good_driver_prob
        
        if is_good:
            # [착한 운전]
            helmet_tm = duration_sec # 헬멧 쓴 시간 = 전체 시간
            nohelmet_cnt = 0         # ★ 수정: 횟수는 0
            score = random.randint(10, 20)
        else:
            # [나쁜 운전]
            helmet_tm = 0
            nohelmet_cnt = 1         # ★ 수정: 안 썼으면 횟수는 1 (초 단위 아님!)
            score = random.randint(-5, -2) # 감점 폭도 조금 줄임

        ride_id = "BATCH_" + ride_time.strftime("%Y%m%d%H%M%S") + f"_{random.randint(100,999)}"
        row = (ride_id, u_id, k_id, ride_time, end_time, helmet_tm, nohelmet_cnt, score, "P")
        data_buffer.append(row)

sql = "INSERT INTO TB_RIDE VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)"
try:
    cursor.executemany(sql, data_buffer)
    conn.commit()
    print(f"✅ 성공! 발표용 데이터 {len(data_buffer)}건 저장 완료.")
except Exception as e:
    print(f"💥 에러: {e}")
finally:
    conn.close()

🔍 데이터 준비 중...
🧹 기존 데이터 삭제...
✨ '발표용 최적화' 데이터 생성 시작 (단위 수정됨)... 🚀
✅ 성공! 발표용 데이터 883건 저장 완료.


# 발표 시연용 영상 만들기

In [12]:
import cv2
from ultralytics import YOLO

# 1. 모델 로드
print("⏳ 모델 로딩 중...")
model = YOLO('data/best (sDUDU).pt') 

# 2. 원본 영상 경로 (편집한 15초 영상 파일명 넣기)
input_path = "data/주간헬멧.mp4" 
output_path = "발표용_최종결과.mp4"

cap = cv2.VideoCapture(input_path)

# 영상 정보 가져오기 (저장할 때 필요)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 3. 영상 저장 설정 (코덱: mp4v)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"🎬 변환 시작! ({width}x{height} @ {fps}fps)")
frame_cnt = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # AI 예측
    results = model(frame)
    
    # ★ 박스가 그려진 화면(annotated_frame)을 가져옴
    annotated_frame = results[0].plot()
    
    # (선택사항) 발표용으로 글씨를 더 크게 넣고 싶으면 여기서 cv2.putText 추가 가능
    # cv2.putText(annotated_frame, "DUDU AI Detecting...", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # 변환된 프레임을 저장
    out.write(annotated_frame)
    
    frame_cnt += 1
    if frame_cnt % 30 == 0:
        print(f"   ... {frame_cnt} 프레임 처리 중")

cap.release()
out.release()
print(f"✅ 변환 완료! 저장된 파일: {output_path}")
print("이제 이 영상을 PPT에 넣거나 발표 때 트시면 됩니다.")

⏳ 모델 로딩 중...
🎬 변환 시작! (406x720 @ 24.0fps)

0: 640x384 1 helmet, 48.5ms
Speed: 2.5ms preprocess, 48.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 44.2ms
Speed: 3.5ms preprocess, 44.2ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 36.7ms
Speed: 1.8ms preprocess, 36.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 38.3ms
Speed: 1.4ms preprocess, 38.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 35.9ms
Speed: 1.7ms preprocess, 35.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 37.7ms
Speed: 1.9ms preprocess, 37.7ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 1 helmet, 34.8ms
Speed: 1.8ms preprocess, 34.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 36.2ms
Speed: 1.8ms p

In [13]:
import cv2
from ultralytics import YOLO

# 1. 모델 로드
print("⏳ 모델 로딩 중...")
model = YOLO('data/best (sDUDU).pt') 

# 2. 원본 영상 경로 (편집한 15초 영상 파일명 넣기)
input_path = "data/야간헬멧.mp4" 
output_path = "발표용_최종결과(2).mp4"

cap = cv2.VideoCapture(input_path)

# 영상 정보 가져오기 (저장할 때 필요)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 3. 영상 저장 설정 (코덱: mp4v)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"🎬 변환 시작! ({width}x{height} @ {fps}fps)")
frame_cnt = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # AI 예측
    results = model(frame)
    
    # ★ 박스가 그려진 화면(annotated_frame)을 가져옴
    annotated_frame = results[0].plot()
    
    # (선택사항) 발표용으로 글씨를 더 크게 넣고 싶으면 여기서 cv2.putText 추가 가능
    # cv2.putText(annotated_frame, "DUDU AI Detecting...", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # 변환된 프레임을 저장
    out.write(annotated_frame)
    
    frame_cnt += 1
    if frame_cnt % 30 == 0:
        print(f"   ... {frame_cnt} 프레임 처리 중")

cap.release()
out.release()
print(f"✅ 변환 완료! 저장된 파일: {output_path}")
print("이제 이 영상을 PPT에 넣거나 발표 때 트시면 됩니다.")

⏳ 모델 로딩 중...
🎬 변환 시작! (406x720 @ 24.0fps)

0: 640x384 1 helmet, 43.1ms
Speed: 1.7ms preprocess, 43.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 38.9ms
Speed: 1.7ms preprocess, 38.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 43.6ms
Speed: 2.0ms preprocess, 43.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 45.1ms
Speed: 1.8ms preprocess, 45.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 40.8ms
Speed: 1.9ms preprocess, 40.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 44.5ms
Speed: 1.8ms preprocess, 44.5ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 43.0ms
Speed: 1.7ms preprocess, 43.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 helmet, 43.7ms
Speed: 1.8ms preprocess, 43.7ms inference, 0.6m

In [14]:
import cv2
from ultralytics import YOLO

# 1. 모델 로드
print("⏳ 모델 로딩 중...")
model = YOLO('data/best (sDUDU).pt') 

# 2. 원본 영상 경로 (편집한 15초 영상 파일명 넣기)
input_path = "data/야간노헬멧.mp4" 
output_path = "발표용_최종결과(야간,노헬멧).mp4"

cap = cv2.VideoCapture(input_path)

# 영상 정보 가져오기 (저장할 때 필요)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 3. 영상 저장 설정 (코덱: mp4v)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"🎬 변환 시작! ({width}x{height} @ {fps}fps)")
frame_cnt = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # AI 예측
    results = model(frame)
    
    # ★ 박스가 그려진 화면(annotated_frame)을 가져옴
    annotated_frame = results[0].plot()
    
    # (선택사항) 발표용으로 글씨를 더 크게 넣고 싶으면 여기서 cv2.putText 추가 가능
    # cv2.putText(annotated_frame, "DUDU AI Detecting...", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # 변환된 프레임을 저장
    out.write(annotated_frame)
    
    frame_cnt += 1
    if frame_cnt % 30 == 0:
        print(f"   ... {frame_cnt} 프레임 처리 중")

cap.release()
out.release()
print(f"✅ 변환 완료! 저장된 파일: {output_path}")
print("이제 이 영상을 PPT에 넣거나 발표 때 트시면 됩니다.")

⏳ 모델 로딩 중...
🎬 변환 시작! (406x720 @ 24.0fps)

0: 640x384 1 no_helmet, 38.8ms
Speed: 2.8ms preprocess, 38.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 42.4ms
Speed: 1.8ms preprocess, 42.4ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 no_helmets, 44.9ms
Speed: 2.0ms preprocess, 44.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 40.0ms
Speed: 1.8ms preprocess, 40.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 no_helmets, 41.3ms
Speed: 3.0ms preprocess, 41.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 42.0ms
Speed: 2.0ms preprocess, 42.0ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 41.1ms
Speed: 2.2ms preprocess, 41.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 41.5ms
Speed: 1.9ms preproce

In [15]:
import cv2
from ultralytics import YOLO

# 1. 모델 로드
print("⏳ 모델 로딩 중...")
model = YOLO('data/best (sDUDU).pt') 

# 2. 원본 영상 경로 (편집한 15초 영상 파일명 넣기)
input_path = "data/주간노헬멧.mp4" 
output_path = "발표용_최종결과(주간,노헬멧).mp4"

cap = cv2.VideoCapture(input_path)

# 영상 정보 가져오기 (저장할 때 필요)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# 3. 영상 저장 설정 (코덱: mp4v)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"🎬 변환 시작! ({width}x{height} @ {fps}fps)")
frame_cnt = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # AI 예측
    results = model(frame)
    
    # ★ 박스가 그려진 화면(annotated_frame)을 가져옴
    annotated_frame = results[0].plot()
    
    # (선택사항) 발표용으로 글씨를 더 크게 넣고 싶으면 여기서 cv2.putText 추가 가능
    # cv2.putText(annotated_frame, "DUDU AI Detecting...", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # 변환된 프레임을 저장
    out.write(annotated_frame)
    
    frame_cnt += 1
    if frame_cnt % 30 == 0:
        print(f"   ... {frame_cnt} 프레임 처리 중")

cap.release()
out.release()
print(f"✅ 변환 완료! 저장된 파일: {output_path}")
print("이제 이 영상을 PPT에 넣거나 발표 때 트시면 됩니다.")

⏳ 모델 로딩 중...
🎬 변환 시작! (406x720 @ 24.0fps)

0: 640x384 1 no_helmet, 38.2ms
Speed: 1.5ms preprocess, 38.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 36.5ms
Speed: 1.5ms preprocess, 36.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 36.8ms
Speed: 1.8ms preprocess, 36.8ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 44.0ms
Speed: 1.6ms preprocess, 44.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 41.0ms
Speed: 1.9ms preprocess, 41.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 43.4ms
Speed: 2.2ms preprocess, 43.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 36.1ms
Speed: 1.4ms preprocess, 36.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 no_helmet, 39.5ms
Speed: 1.8ms preprocess